In [ ]:


!pip install -q tensorflow numpy pandas scikit-learn opencv-python-headless tqdm

import os, gc, json, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)
from tqdm.auto import tqdm

np.random.seed(0)
tf.random.set_seed(0)



In [ ]:
# ============================================================
# 1. ACCELERATOR DETECTION — TPU v5e-1 -> GPU (T4) -> CPU
# ============================================================
STRATEGY = None
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    STRATEGY = tf.distribute.TPUStrategy(resolver)
    print(f"✅ Using TPU: {resolver.master()} | replicas: {STRATEGY.num_replicas_in_sync}")
except Exception:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for g in gpus:
            try:
                tf.config.experimental.set_memory_growth(g, True)
            except Exception:
                pass
        STRATEGY = tf.distribute.get_strategy()
        print(f"✅ Using GPU: {[g.name for g in gpus]}")
    else:
        STRATEGY = tf.distribute.get_strategy()
        print("⚠️ No TPU/GPU found — using CPU (will be slow).")



✅ Using GPU: ['/physical_device:GPU:0']


In [ ]:
# ============================================================
# 2. GOOGLE DRIVE + PROJECT PATHS
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# >>> EDIT if your NeoJaundice folder lives elsewhere in Drive <
NEOJAUNDICE_ROOT = Path("/content/drive/MyDrive/Neojaundice_main/NeoJaundice/NeoJaundice")

IMAGES_DIR = NEOJAUNDICE_ROOT / "images"
CSV_PATH   = NEOJAUNDICE_ROOT / "chd_jaundice_published_2.csv"
MODEL_DIR   = NEOJAUNDICE_ROOT / "models"
RESULTS_DIR = NEOJAUNDICE_ROOT / "results"
for d in (MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"images/ not found at {IMAGES_DIR}")
if not CSV_PATH.exists():
    # fall back to any single csv in the root, in case the filename differs slightly
    csvs = list(NEOJAUNDICE_ROOT.glob("*.csv"))
    if len(csvs) == 1:
        CSV_PATH = csvs[0]
        print(f"⚠️ Expected filename not found — using the only CSV present: {CSV_PATH.name}")
    else:
        raise FileNotFoundError(f"Labels CSV not found at {CSV_PATH}, and could not uniquely guess a replacement.")

print(f"Images dir : {IMAGES_DIR}")
print(f"Labels CSV : {CSV_PATH}")

Mounted at /content/drive
Images dir : /content/drive/MyDrive/Neojaundice_main/NeoJaundice/NeoJaundice/images
Labels CSV : /content/drive/MyDrive/Neojaundice_main/NeoJaundice/NeoJaundice/chd_jaundice_published_2.csv


In [ ]:
# ============================================================
# STAGE 2: metadata baseline + 2D transfer learning (EfficientNetB0) + metadata branch
# Run this in NEW cells AFTER the previous script (needs: data, labels_df, STRATEGY, RESULTS_DIR,
# color_correct_with_card, extract_skin_roi, holdout_split, CSV_GROUP_COL, META_COLS, COLOR_SPACES, color_stats)
# ============================================================
import gc, cv2, numpy as np, pandas as pd, tensorflow as tf
from concurrent.futures import ThreadPoolExecutor
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------
TASK             = "binary"   # "binary" (bilirubin > BINARY_THRESHOLD) or "3class" (0-5 / 5-10 / >10)
BINARY_THRESHOLD = 12.9
IMG              = 128        # ROI size for the 2D model (try 160 or 224 later)
USE_META         = True       # feed age/gestational age/weight/gender into the model
USE_CLASS_WEIGHT = False
N_FOLDS          = 5
EPOCHS_HEAD, EPOCHS_FT = 20, 20
BATCH            = 32

# ---------------- LABELS ----------------
if TASK == "binary":
    y_all = (data.bilirubin > BINARY_THRESHOLD).astype(np.int64)
    n_classes = 2
else:
    y_all = data.jaundice_label.astype(np.int64)
    n_classes = 3
groups = data.meta[CSV_GROUP_COL].values
print("Class counts:", dict(zip(*np.unique(y_all, return_counts=True))),
      "| majority baseline:", round(np.bincount(y_all).max() / len(y_all), 3))

# ---------------- METADATA MATRIX ----------------
meta = data.meta
M_df = pd.DataFrame({
    "age":  pd.to_numeric(meta["age(day)"], errors="coerce"),
    "ga":   pd.to_numeric(meta["gestational_age"], errors="coerce"),
    "wt":   pd.to_numeric(meta["weight"], errors="coerce"),
    "male": (meta["gender"].astype(str).str.strip().str.upper() == "M").astype(float),
})
M_all = M_df.fillna(M_df.median()).to_numpy(np.float32)

# ---------------- CV SPLITTER (patient-level) ----------------
def cv_splits(seed=0):
    outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for trainval, test in outer.split(np.zeros(len(y_all)), y_all, groups):
        inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=seed)
        tr, va = next(inner.split(trainval, y_all[trainval], groups[trainval]))
        yield trainval[tr], trainval[va], test

def report(name, y_true, y_pred, y_prob=None):
    out = {"acc": accuracy_score(y_true, y_pred),
           "bal_acc": balanced_accuracy_score(y_true, y_pred),
           "macroF1": f1_score(y_true, y_pred, average="macro")}
    if y_prob is not None:
        try:
            out["AUC"] = roc_auc_score(y_true, y_prob[:, 1]) if n_classes == 2 \
                else roc_auc_score(y_true, y_prob, multi_class="ovr")
        except ValueError:
            pass
    print(f"[{name}] " + "  ".join(f"{k}={v:.3f}" for k, v in out.items()))
    print("  confusion matrix:\n", confusion_matrix(y_true, y_pred))
    return out

# ============================================================
# A. TABULAR BASELINES (pooled out-of-fold, patient-level)
#    metadata only  vs  metadata + color stats.  This tells you the ceiling before any CNN.
# ============================================================
import lightgbm as lgb

X_color = np.concatenate([color_stats(data.features[cs]) for cs in COLOR_SPACES], axis=1)
for name, X in [("LGBM metadata only", M_all),
                ("LGBM color only", X_color),
                ("LGBM metadata + color", np.hstack([M_all, X_color]))]:
    oof_pred = np.zeros(len(y_all), dtype=int)
    oof_prob = np.zeros((len(y_all), n_classes))
    for tr, va, te in cv_splits(seed=0):
        clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                 min_child_samples=20, verbose=-1)
        clf.fit(X[tr], y_all[tr])
        oof_prob[te] = clf.predict_proba(X[te])
        oof_pred[te] = oof_prob[te].argmax(1)
    report(name, y_all, oof_pred, oof_prob)

# ============================================================
# B. BUILD 2D ROI ARRAY (cached)
# ============================================================
IMG_CACHE = RESULTS_DIR / f"img2d_{IMG}_n{len(y_all)}.npy"

def load_roi_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    corrected = color_correct_with_card(img)
    roi = extract_skin_roi(corrected, roi_size=(IMG, IMG))
    return cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)          # uint8, 0-255

if IMG_CACHE.exists():
    X_img = np.load(IMG_CACHE)
    print("Loaded cached 2D ROIs:", X_img.shape)
else:
    paths = data.meta["resolved_path"].tolist()
    with ThreadPoolExecutor(max_workers=8) as ex:
        X_img = np.stack(list(tqdm(ex.map(load_roi_rgb, paths), total=len(paths), desc="2D ROIs")))
    np.save(IMG_CACHE, X_img)
    print("Saved 2D ROIs:", X_img.shape)

# ============================================================
# C. MODEL: EfficientNetB0 (+ metadata branch)
#    No color jitter on purpose: jaundice IS a color signal. Only flips / small rotations.
# ============================================================
def build_model():
    with STRATEGY.scope():
        img_in = layers.Input((IMG, IMG, 3), name="img")
        x = layers.RandomFlip("horizontal_and_vertical")(img_in)
        x = layers.RandomRotation(0.05)(x)
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                    input_shape=(IMG, IMG, 3), pooling="avg")
        base.trainable = False
        f = base(x, training=False)                     # EfficientNet expects raw 0-255 pixels
        f = layers.Dropout(0.3)(f)
        inputs = [img_in]
        if USE_META:
            m_in = layers.Input((M_all.shape[1],), name="meta")
            m = layers.Dense(32, activation="relu")(m_in)
            m = layers.Dense(32, activation="relu")(m)
            f = layers.Concatenate()([f, m])
            inputs.append(m_in)
        f = layers.Dense(64, activation="relu")(f)
        f = layers.Dropout(0.3)(f)
        out = layers.Dense(n_classes, activation="softmax")(f)
        model = models.Model(inputs, out)
    return model, base

def compile_model(model, lr):
    with STRATEGY.scope():
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def make_inputs(idx, mu, sd):
    d = {"img": X_img[idx].astype(np.float32)}
    if USE_META:
        d["meta"] = (M_all[idx] - mu) / sd
    return d

# ============================================================
# D. PATIENT-LEVEL CROSS-VALIDATION
# ============================================================
oof_pred = np.zeros(len(y_all), dtype=int)
oof_prob = np.zeros((len(y_all), n_classes))
fold_acc = []

for fold, (tr, va, te) in enumerate(cv_splits(seed=0)):
    tf.keras.backend.clear_session()
    tf.random.set_seed(fold)
    mu, sd = M_all[tr].mean(0, keepdims=True), M_all[tr].std(0, keepdims=True) + 1e-6
    model, base = build_model()

    cw = None
    if USE_CLASS_WEIGHT:
        cls = np.unique(y_all[tr])
        cw = dict(zip(cls.tolist(), compute_class_weight("balanced", classes=cls, y=y_all[tr]).tolist()))

    cbs = lambda: [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                   tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7)]
    Xtr, Xva, Xte = make_inputs(tr, mu, sd), make_inputs(va, mu, sd), make_inputs(te, mu, sd)

    # Phase 1: train the head only
    compile_model(model, 1e-3)
    model.fit(Xtr, y_all[tr], validation_data=(Xva, y_all[va]), epochs=EPOCHS_HEAD,
              batch_size=BATCH, verbose=0, class_weight=cw, callbacks=cbs())

    # Phase 2: fine-tune top of the backbone (BatchNorm layers stay frozen)
    base.trainable = True
    for l in base.layers[:-30]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    compile_model(model, 1e-5)
    model.fit(Xtr, y_all[tr], validation_data=(Xva, y_all[va]), epochs=EPOCHS_FT,
              batch_size=BATCH, verbose=0, class_weight=cw, callbacks=cbs())

    prob = model.predict(Xte, batch_size=BATCH, verbose=0)
    oof_prob[te] = prob
    oof_pred[te] = prob.argmax(1)
    acc = accuracy_score(y_all[te], oof_pred[te])
    fold_acc.append(acc)
    print(f"fold {fold}: acc={acc:.3f}  (test majority baseline={np.bincount(y_all[te]).max()/len(te):.3f})")
    del model, base
    gc.collect()

print(f"\nMean fold accuracy: {np.mean(fold_acc):.3f} ± {np.std(fold_acc):.3f}")
report(f"EfficientNetB0 {'+meta' if USE_META else 'image only'} [{TASK}]", y_all, oof_pred, oof_prob)


Class counts: {np.int64(0): np.int64(1299), np.int64(1): np.int64(933)} | majority baseline: 0.582


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LGBM metadata only] acc=0.673  bal_acc=0.665  macroF1=0.665  AUC=0.737
  confusion matrix:
 [[930 369]
 [360 573]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LGBM color only] acc=0.754  bal_acc=0.750  macroF1=0.749  AUC=0.830
  confusion matrix:
 [[1011  288]
 [ 260  673]]


...

[LGBM metadata + color] acc=0.773  bal_acc=0.770  macroF1=0.768  AUC=0.852
  confusion matrix:
 [[1026  273]
 [ 233  700]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


2D ROIs:   0%|          | 0/2232 [00:00<?, ?it/s]

Saved 2D ROIs: (2232, 128, 128, 3)
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
fold 0: acc=0.720  (test majority baseline=0.604)
fold 1: acc=0.709  (test majority baseline=0.537)
fold 2: acc=0.797  (test majority baseline=0.588)
fold 3: acc=0.705  (test majority baseline=0.577)
fold 4: acc=0.705  (test majority baseline=0.604)

Mean fold accuracy: 0.727 ± 0.035
[EfficientNetB0 +meta [binary]] acc=0.727  bal_acc=0.714  macroF1=0.716  AUC=0.792
  confusion matrix:
 [[1031  268]
 [ 341  592]]


{'acc': 0.7271505376344086,
 'bal_acc': np.float64(0.71409988885836),
 'macroF1': 0.7161706482403907,
 'AUC': np.float64(0.7917963112857034)}

In [ ]:
# ============================================================
# STAGE 3: richer color features + LGBM/LogReg ensemble + CNN blend + patient-level averaging
# Run in NEW cells right after stage 2 (needs: X_img, data, y_all, groups, M_all, n_classes, IMG,
# cv_splits, oof_prob (CNN out-of-fold probs from stage 2), extract_skin_roi,
# find_yellow_card_mask, largest_contour_bbox)
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import cv2, numpy as np, pandas as pd
import lightgbm as lgb
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

cnn_oof_prob = oof_prob.copy()   # CNN out-of-fold probabilities (seed-0 folds) from stage 2.
                                 # Run this cell right after stage 2 so oof_prob is still the CNN's.

# ---------------- 1. Rich color features from the calibrated 128x128 ROI ----------------
def stats7(a):                   # a: (n_pixels, C) -> mean, std, p10/p25/p50/p75/p90 per channel
    q = np.percentile(a, [10, 25, 50, 75, 90], axis=0)
    return np.concatenate([a.mean(0), a.std(0), q.ravel()])

def roi_features(rgb):           # rgb: uint8 (H, W, 3), already card-corrected
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    spaces = [rgb.astype(np.float32) / 255.0,
              cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32),
              cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32),
              cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb).astype(np.float32)]
    feats = [stats7(s.reshape(-1, 3)) for s in spaces]

    R, G, B = [rgb[..., i].astype(np.float32).ravel() + 1.0 for i in range(3)]
    ratios = [R / G, B / G, (R - B) / (R + B), R / (R + G + B), G / (R + G + B), B / (R + G + B)]
    feats.append(np.array([r.mean() for r in ratios] + [np.median(r) for r in ratios], np.float32))

    lab = spaces[2].reshape(-1, 3)
    L, a, b = lab[:, 0] + 1.0, lab[:, 1] - 128.0, lab[:, 2] - 128.0
    feats.append(np.array([np.mean(b / L), np.median(b / L),
                           np.mean(np.sqrt(a * a + b * b)), np.mean(np.arctan2(b, a))], np.float32))

    # skin-only pixels (YCrCb rule; OpenCV order is Y, Cr, Cb); fall back to all pixels if too few
    ycc = spaces[3].reshape(-1, 3)
    skin = (ycc[:, 1] >= 133) & (ycc[:, 1] <= 173) & (ycc[:, 2] >= 77) & (ycc[:, 2] <= 127)
    if skin.sum() < 200:
        skin = np.ones(len(ycc), bool)
    feats.append(np.array([skin.mean()], np.float32))
    feats.append(np.median(lab[skin], 0))
    feats.append(np.median(spaces[0].reshape(-1, 3)[skin], 0))
    return np.concatenate(feats).astype(np.float32)

# ---------------- 2. Card + RAW (uncorrected) features: lets the model learn its own lighting correction ----------------
def extra_features(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    bbox = largest_contour_bbox(find_yellow_card_mask(img))
    lab_full = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    if bbox is None:
        card = np.array([np.nan, np.nan, np.nan, 0.0, 0.0], np.float32)
    else:
        x, y, w, h = bbox
        card_mean = lab_full[y:y + h, x:x + w].reshape(-1, 3).mean(0)
        card = np.concatenate([card_mean, [1.0, (w * h) / (img.shape[0] * img.shape[1])]]).astype(np.float32)
    roi = extract_skin_roi(img, roi_size=(IMG, IMG))                    # RAW ROI, no correction
    raw = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB).astype(np.float32).reshape(-1, 3)
    return np.concatenate([card, np.median(raw, 0), raw.mean(0)]).astype(np.float32)

F_roi = np.stack([roi_features(im) for im in tqdm(X_img, desc="ROI features")])
with ThreadPoolExecutor(max_workers=8) as ex:
    F_extra = np.stack(list(tqdm(ex.map(extra_features, data.meta["resolved_path"].tolist()),
                                 total=len(X_img), desc="card/raw features")))
print("Feature shapes:", M_all.shape, F_roi.shape, F_extra.shape)
print("Card detected in", int(np.nansum(F_extra[:, 3])), "of", len(F_extra), "images")

# ---------------- 3. Models + evaluation helpers ----------------
def make_lgbm():
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.02, num_leaves=7,
                              min_child_samples=30, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.5, reg_lambda=5.0, verbose=-1)

def make_lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=0.05, max_iter=3000))

def oof_from(make, X, seed=0):
    prob = np.zeros((len(y_all), n_classes))
    for tr, va, te in cv_splits(seed):            # patient-level folds
        m = make()
        m.fit(X[tr], y_all[tr])
        prob[te] = m.predict_proba(X[te])
    return prob

def patient_avg(prob):
    df = pd.DataFrame(prob)
    df["pid"] = groups
    return df.groupby("pid").transform("mean").to_numpy()

def evaluate(name, prob):
    pa = patient_avg(prob)
    auc = roc_auc_score(y_all, prob[:, 1]) if n_classes == 2 else float("nan")
    print(f"{name:38s} per-image acc={accuracy_score(y_all, prob.argmax(1)):.3f}  AUC={auc:.3f}"
          f"  | patient-averaged acc={accuracy_score(y_all, pa.argmax(1)):.3f}")

# ---------------- 4. Run ----------------
XA = np.hstack([M_all, F_roi])
XB = np.hstack([M_all, F_roi, F_extra])
res = {}
print("\n=== Patient-level out-of-fold results (fold seed 0) ===")
evaluate("CNN (stage 2)", cnn_oof_prob)
for fs_name, X in [("A: meta+ROI", XA), ("B: meta+ROI+card/raw", XB)]:
    for mname, mk in [("LGBM", make_lgbm), ("LogReg", make_lr)]:
        res[(fs_name, mname)] = oof_from(mk, X, seed=0)
        evaluate(f"{mname} | {fs_name}", res[(fs_name, mname)])

B_l, B_r = res[("B: meta+ROI+card/raw", "LGBM")], res[("B: meta+ROI+card/raw", "LogReg")]
evaluate("LGBM + LogReg (B)", (B_l + B_r) / 2)
evaluate("LGBM + LogReg + CNN (B)", (B_l + B_r + cnn_oof_prob) / 3)

# repeat with different patient-fold partitions and average (each prediction is still out-of-fold)
ens = [(oof_from(make_lgbm, XB, s) + oof_from(make_lr, XB, s)) / 2 for s in range(3)]
evaluate("LGBM + LogReg (B), 3 fold-seeds avg", np.mean(ens, axis=0))

ROI features:   0%|          | 0/2232 [00:00<?, ?it/s]

card/raw features:   0%|          | 0/2232 [00:00<?, ?it/s]

Feature shapes: (2232, 4) (2232, 107) (2232, 11)
Card detected in 2232 of 2232 images

=== Patient-level out-of-fold results (fold seed 0) ===
CNN (stage 2)                          per-image acc=0.727  AUC=0.792  | patient-averaged acc=0.754
LGBM | A: meta+ROI                     per-image acc=0.772  AUC=0.859  | patient-averaged acc=0.806
LogReg | A: meta+ROI                   per-image acc=0.774  AUC=0.853  | patient-averaged acc=0.819
LGBM | B: meta+ROI+card/raw            per-image acc=0.785  AUC=0.867  | patient-averaged acc=0.812
LogReg | B: meta+ROI+card/raw          per-image acc=0.779  AUC=0.863  | patient-averaged acc=0.828
LGBM + LogReg (B)                      per-image acc=0.795  AUC=0.875  | patient-averaged acc=0.827
LGBM + LogReg + CNN (B)                per-image acc=0.789  AUC=0.878  | patient-averaged acc=0.819
LGBM + LogReg (B), 3 fold-seeds avg    per-image acc=0.797  AUC=0.876  | patient-averaged acc=0.831


In [ ]:
# ============================================================
# STAGE 4: diagnostics + last honest levers
# Run right after stage 3 (needs: XB, F_extra, data, y_all, groups, n_classes, cv_splits,
# make_lgbm, make_lr, ens, patient_avg, BINARY_THRESHOLD)
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve,
                             mean_squared_error, r2_score)

bili = data.bilirubin.astype(np.float64)
p_ens = np.mean(ens, axis=0)            # best classifier from stage 3 (LGBM+LogReg, 3 fold-seeds)
pred_ens = p_ens.argmax(1)

# ------------------------------------------------------------
# 1. Is the "card" really a card?  (100% detection is suspicious)
# ------------------------------------------------------------
print("=== 1. Card detection sanity ===")
area = F_extra[:, 4]                                    # bbox area / image area
print("card bbox area fraction percentiles [1,5,25,50,75,95,99]:",
      np.round(np.percentile(area, [1, 5, 25, 50, 75, 95, 99]), 4))
print(f"bbox < 0.5% of image: {(area < 0.005).mean():.1%} | bbox > 30% of image: {(area > 0.30).mean():.1%}")
for i, nm in enumerate(["card L", "card a", "card b"]):
    print(f"corr({nm}, bilirubin) = {np.corrcoef(F_extra[:, i], bili)[0, 1]:+.3f}")
print("If the card color correlates strongly with bilirubin, the 'card' is probably skin/background,\n"
      "and the color correction is erasing part of the signal. Check preprocessing_preview.png.")

# ------------------------------------------------------------
# 2. Where do the errors come from?  Accuracy vs distance to the 12.9 mg/dL cutoff
# ------------------------------------------------------------
print("\n=== 2. Accuracy by |TSB - threshold| (best classifier) ===")
d = np.abs(bili - BINARY_THRESHOLD)
edges = [0, 1, 2, 3, 5, 100]
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (d >= lo) & (d < hi)
    if m.sum():
        print(f"  [{lo:>3}, {hi:>3}) mg/dL from cutoff: n={m.sum():4d}  acc={accuracy_score(y_all[m], pred_ens[m]):.3f}")
print("Errors concentrated near the cutoff = label ambiguity, not a modeling bug.")

# ------------------------------------------------------------
# 3. Regression, then threshold (uses the continuous label)
# ------------------------------------------------------------
print("\n=== 3. Regression then threshold ===")
def make_lgbm_reg():
    return lgb.LGBMRegressor(n_estimators=500, learning_rate=0.02, num_leaves=7, min_child_samples=30,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, verbose=-1)
def make_ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=100.0))

def oof_reg(make, X, seed=0):
    pred = np.zeros(len(bili))
    for tr, va, te in cv_splits(seed):
        m = make(); m.fit(X[tr], bili[tr]); pred[te] = m.predict(X[te])
    return pred

reg_preds = {}
for nm, mk in [("LGBM-reg", make_lgbm_reg), ("Ridge", make_ridge)]:
    reg_preds[nm] = np.mean([oof_reg(mk, XB, s) for s in range(3)], axis=0)
reg_preds["LGBM+Ridge"] = (reg_preds["LGBM-reg"] + reg_preds["Ridge"]) / 2

for nm, p in reg_preds.items():
    pa = pd.Series(p).groupby(groups).transform("mean").to_numpy()
    print(f"  {nm:11s} RMSE={np.sqrt(mean_squared_error(bili, p)):.2f}  R2={r2_score(bili, p):.3f}  "
          f"acc@12.9={accuracy_score(y_all, (p > BINARY_THRESHOLD).astype(int)):.3f}  "
          f"AUC={roc_auc_score(y_all, p):.3f}  patient-avg acc={accuracy_score(y_all, (pa > BINARY_THRESHOLD).astype(int)):.3f}")

# ------------------------------------------------------------
# 4. Clinical-style operating points (what a screening paper would report)
# ------------------------------------------------------------
print("\n=== 4. Operating points (best classifier, out-of-fold) ===")
score = p_ens[:, 1]
fpr, tpr, thr = roc_curve(y_all, score)
for target in (0.90, 0.95):
    i = int(np.argmax(tpr >= target))
    print(f"  sensitivity >= {target:.0%}: specificity = {1 - fpr[i]:.3f} (threshold {thr[i]:.3f})")
pa_score = pd.Series(score).groupby(groups).transform("mean").to_numpy()
print(f"  AUC per-image = {roc_auc_score(y_all, score):.3f} | patient-averaged = {roc_auc_score(y_all, pa_score):.3f}")
print("  (thresholds are chosen on out-of-fold scores, so treat them as optimistic estimates.)")

# ------------------------------------------------------------
# 5. DIAGNOSTIC ONLY: what an image-level (leaky) split would report
#    Do NOT report this number as performance. It shows how a random split can inflate accuracy,
#    which is worth checking against whatever paper set your 90% target.
# ------------------------------------------------------------
print("\n=== 5. DIAGNOSTIC ONLY: image-level random split (leaky) ===")
prob_leaky = np.zeros((len(y_all), n_classes))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for tr, te in skf.split(XB, y_all):
    ml, mr = make_lgbm(), make_lr()
    ml.fit(XB[tr], y_all[tr]); mr.fit(XB[tr], y_all[tr])
    prob_leaky[te] = (ml.predict_proba(XB[te]) + mr.predict_proba(XB[te])) / 2
print(f"  leaky image-level acc = {accuracy_score(y_all, prob_leaky.argmax(1)):.3f}   "
      f"AUC = {roc_auc_score(y_all, prob_leaky[:, 1]):.3f}")
print(f"  honest patient-level acc = {accuracy_score(y_all, pred_ens):.3f}   "
      f"AUC = {roc_auc_score(y_all, p_ens[:, 1]):.3f}")

=== 1. Card detection sanity ===
card bbox area fraction percentiles [1,5,25,50,75,95,99]: [0.0456 0.0493 0.0542 0.058  0.0618 0.069  0.0762]
bbox < 0.5% of image: 0.0% | bbox > 30% of image: 0.0%
corr(card L, bilirubin) = +0.037
corr(card a, bilirubin) = -0.079
corr(card b, bilirubin) = -0.132
If the card color correlates strongly with bilirubin, the 'card' is probably skin/background,
and the color correction is erasing part of the signal. Check preprocessing_preview.png.

=== 2. Accuracy by |TSB - threshold| (best classifier) ===
  [  0,   1) mg/dL from cutoff: n= 309  acc=0.550
  [  1,   2) mg/dL from cutoff: n= 309  acc=0.654
  [  2,   3) mg/dL from cutoff: n= 312  acc=0.766
  [  3,   5) mg/dL from cutoff: n= 522  acc=0.810
  [  5, 100) mg/dL from cutoff: n= 780  acc=0.956
Errors concentrated near the cutoff = label ambiguity, not a modeling bug.

=== 3. Regression then threshold ===
  LGBM-reg    RMSE=2.96  R2=0.676  acc@12.9=0.793  AUC=0.874  patient-avg acc=0.820
  Ridge       